# TP 4 — Apprentissage supervise 2A (correction R)

**Deux parties :**
- **Partie 1 (TD)** : matrices de confusion et courbes ROC, a partir d'un tableau de scores fourni.
- **Partie 2 (TP)** : detection de **spam** (jeu `spambase`), comparaison de plusieurs modeles via **ROC / AUC**.

> Correction en **R** : `MASS` (LDA/QDA), `rpart` (CART + cout differencie), `ranger` (foret), `caret` (k-NN
> standardise + validation croisee), `pROC` (ROC/AUC) et **plotly**. Le jeu `spam` provient du package `kernlab`.


## Brief de demarrage — Les outils de ce TP

> **A lire avant de commencer.** Ce TP ne porte pas sur un nouveau modele mais sur la facon d'**evaluer et de
> comparer** des classifieurs quand le seuil de decision compte (detection de spam). Voici les outils cles.

**1. Matrice de confusion — le point de depart.**
Pour un seuil donne, on croise verite et prediction :

| | predit negatif | predit positif |
|---|---|---|
| **reel negatif** | VN | FP |
| **reel positif** | FN | VP |

On en tire notamment :
- **Sensibilite / rappel** (TVP) $= \frac{VP}{VP+FN}$ : part des positifs bien detectes ;
- **Specificite** $= \frac{VN}{VN+FP}$ ; **taux de faux positifs** (TFP) $= 1 - \text{specificite}$ ;
- **Precision** $= \frac{VP}{VP+FP}$.

**2. Courbe ROC et AUC — evaluer *independamment du seuil*.**
Un classifieur produit un **score** $s(x)$. En faisant **varier le seuil**, on parcourt tous les compromis
(TVP en fonction du TFP) : c'est la **courbe ROC**. L'**AUC** (aire sous la courbe) resume la capacite du modele
a **ordonner** les positifs avant les negatifs : $0.5$ = aleatoire, $1$ = parfait. L'AUC ne depend **pas** du
seuil choisi, ce qui la rend ideale pour **comparer des modeles**.

**3. Couts asymetriques et choix du seuil.**
En pratique, une erreur peut couter plus cher qu'une autre (laisser passer un spam vs bloquer un vrai message).
On **deplace le seuil** (ou on pondere les classes) pour privilegier la sensibilite ou la specificite selon le
cout metier. La courbe ROC aide a **choisir le point de fonctionnement**.

> **Methode du TP.** (Partie 1) construire matrices de confusion et courbe ROC a la main depuis des scores
> fournis ; (Partie 2) entrainer plusieurs classifieurs (LDA/QDA, CART, foret, k-NN...) sur `spambase` et les
> **comparer via ROC / AUC**.

In [1]:
source(if (file.exists("plot_theme.R")) "plot_theme.R" else "../plot_theme.R")  # theme + charger_packages()
charger_packages(c(
                   "kernlab",
                   "MASS",
                   "rpart",
                   "ranger",
                   "caret",
                   "pROC",
                   "plotly"
))

SEED <- 0
set.seed(SEED)   # graine unique posee une fois : tout le TP est reproductible

## Partie 1 — Matrices de confusion, courbes ROC

### Rappel de cours

On est en discrimination binaire, l'evenement d'interet est $Y = 1$. Un classifieur produit un **score**
$s(x) \in [0,1]$ (estimation de $\mathbb{P}(Y=1\mid x)$) ; on decide $\hat{Y} = 1$ si $s(x) \ge$ **seuil**.

**Matrice de confusion** (a seuil fixe) :

|               | Predit 1 | Predit 0 |
|---------------|----------|----------|
| **Reel 1**    | VP (vrai positif) | FN (faux negatif) |
| **Reel 0**    | FP (faux positif) | VN (vrai negatif) |

- **Sensibilite** (taux de vrais positifs, TPR) : $\text{TPR} = \dfrac{VP}{VP + FN}$ ;
- **Specificite** : $\dfrac{VN}{VN + FP}$, et **taux de faux positifs** $\text{FPR} = 1 - \text{specificite}$.

**Courbe ROC** : on fait **varier le seuil** de $1$ vers $0$ et on trace le point $(\text{FPR}, \text{TPR})$.
Elle decrit le compromis entre detecter les positifs et eviter les fausses alertes, **independamment du seuil**.


### Question 1 — Construction de la matrice de confusion

A seuil fixe, on compare la prediction $\hat{Y} = \mathbb{1}\{s(x) \ge \text{seuil}\}$ a la verite $Y$, et on
compte les 4 cas (VP, FP, VN, FN). Chaque seuil donne **une** matrice de confusion.


### Question 2 — Donnees fournies et trace de la ROC

**2.1** Le **score** est la probabilite estimee $\hat{\mathbb{P}}(Y=1\mid x)$. La colonne « Individual » est
l'**identifiant** de l'observation ; « Observation Y » est la **vraie classe** ($0$ ou $1$). On saisit les 27
observations de l'enonce.


In [2]:

donnees <- data.frame(
  Individual = 1:27,
  Y = c(1,0,1,1,1,1,0,0,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0),
  Score = c(0.92825,0.82762,0.74119,0.73182,0.72418,0.63537,0.63155,0.61763,
            0.49700,0.46686,0.37062,0.31244,0.29258,0.26446,0.22945,0.21983,
            0.17818,0.13490,0.12012,0.05337,0.01124,0.00639,0.00241,0.00102,
            0.00099,0.00040,0.00011)
)
P <- sum(donnees$Y == 1); N <- sum(donnees$Y == 0)
cat(sprintf("Nombre de positifs P = %d | negatifs N = %d\n", P, N))
head(donnees)


Nombre de positifs P = 9 | negatifs N = 18


,Individual,Y,Score
,<int>,<dbl>,<dbl>
1,1,1,0.92825
2,2,0,0.82762
3,3,1,0.74119
4,4,1,0.73182
5,5,1,0.72418
6,6,1,0.63537


**2.2** L'**axe des x** est le **taux de faux positifs** (FPR), l'**axe des y** le **taux de vrais positifs**
(TPR). On abaisse progressivement le seuil : a chaque score, on calcule (FPR, TPR) et on relie les points.

**2.3** On trie par score **decroissant** et on ajoute les colonnes cumulees **VP** et **FP**, d'ou **TPR** et **FPR**.


In [3]:

tab <- donnees[order(-donnees$Score), ]
tab$VP_cumule <- cumsum(tab$Y == 1)   # vrais positifs si seuil = ce score
tab$FP_cumule <- cumsum(tab$Y == 0)   # faux positifs
tab$TPR <- tab$VP_cumule / P
tab$FPR <- tab$FP_cumule / N
tab[, c("Individual", "Y", "Score", "VP_cumule", "FP_cumule", "TPR", "FPR")]


,Individual,Y,Score,VP_cumule,FP_cumule,TPR,FPR
,<int>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>
1,1,1,0.92825,1,0,0.1111111,0.00000000
2,2,0,0.82762,1,1,0.1111111,0.05555556
3,3,1,0.74119,2,1,0.2222222,0.05555556
4,4,1,0.73182,3,1,0.3333333,0.05555556
5,5,1,0.72418,4,1,0.4444444,0.05555556
6,6,1,0.63537,5,1,0.5555556,0.05555556
7,7,0,0.63155,5,2,0.5555556,0.11111111
8,8,0,0.61763,5,3,0.5555556,0.16666667
9,9,1,0.49700,6,3,0.6666667,0.16666667


In [4]:

fpr <- c(0, tab$FPR); tpr <- c(0, tab$TPR)   # on part de (0,0)
plot_ly() %>%
  add_trace(x = fpr, y = tpr, type = "scatter", mode = "lines+markers",
            line = list(shape = "hv", width = 3, color = pick_color(0)),
            marker = list(size = 8, color = pick_color(2)), name = "Modele evalue") %>%
  layout(title = "Courbe ROC",
         xaxis = list(title = "Taux de faux positifs (FPR)"),
         yaxis = list(title = "Taux de vrais positifs (TPR)")) %>%
  apply_dark_theme()


HTML widgets cannot be represented in plain text (need html)

**2.4** On ajoute :
- le **modele parfait** : il classe tous les positifs avant tous les negatifs $\rightarrow$ la courbe monte a
  $(0,1)$ puis va a $(1,1)$ ;
- le **modele aleatoire** : la **diagonale** $y = x$ (AUC $= 0.5$).


In [5]:
auc_td <- as.numeric(roc(donnees$Y, donnees$Score, quiet = TRUE)$auc)
plot_ly() %>%
  add_trace(x = fpr, y = tpr, type = "scatter", mode = "lines+markers",
            line = list(shape = "hv", width = 3, color = pick_color(0)),
            marker = list(size = 8, color = pick_color(2)), name = "Modele evalue") %>%
  add_trace(x = c(0, 0, 1), y = c(0, 1, 1), type = "scatter", mode = "lines",
            line = list(width = 3, color = pick_color(3)), name = "Modele parfait") %>%
  add_trace(x = c(0, 1), y = c(0, 1), type = "scatter", mode = "lines",
            line = list(color = MUTED, dash = "dash"), name = "Modele aleatoire") %>%
  layout(title = "Courbe ROC : modele evalue, parfait et aleatoire",
         xaxis = list(title = "FPR"), yaxis = list(title = "TPR")) %>%
  apply_dark_theme()

cat("AUC du modele evalue :", round(auc_td, 3), "\n")

HTML widgets cannot be represented in plain text (need html)

AUC du modele evalue : 0.883 


### Question 3 — Analyse de la courbe ROC

**3.1 Points les plus interessants.** Ceux du **coin superieur gauche** (FPR faible, TPR eleve). On choisit
souvent le seuil qui **maximise** $\text{TPR} - \text{FPR}$ (indice de **Youden**), le point le plus eloigne de
la diagonale.

**3.2 Aire sous la courbe (AUC).** C'est la **probabilite que le modele attribue un score plus eleve a un
positif tire au hasard qu'a un negatif tire au hasard** : une mesure de la **qualite de l'ordonnancement**,
entre **0.5** (aleatoire) et **1** (parfait).

**3.3 Comparer deux modeles.** On compare leurs **AUC** ; le plus grand est globalement meilleur. Si les courbes
se **croisent**, le meilleur choix depend du seuil / du cout des erreurs.


## Partie 2 — Detection de spam

### Rappel de cours — Comparaison de modeles et couts differencies

On entraine plusieurs classifieurs et on les compare par **ROC / AUC** (metrique de separation independante du
seuil). Point specifique ici : le **cout differencie** (5 pour 1). Toutes les erreurs ne se valent pas (laisser
passer un spam vs bloquer un email legitime). On l'integre via une **matrice de cout** (`rpart`, argument
`parms = list(loss = ...)`) : penaliser 5x plus les erreurs sur une classe.

Le jeu `spam` (kernlab) : **4601** emails, **57** variables (frequences de mots/caracteres, longueurs de
sequences de majuscules), cible **`spam`** / **`nonspam`**.


In [6]:
data(spam)
y <- factor(ifelse(spam$type == "spam", "spam", "nonspam"), levels = c("nonspam", "spam"))
X <- spam[, setdiff(names(spam), "type")]
cat(sprintf("Dimensions : %d x %d | proportion de spam : %.3f\n", nrow(X), ncol(X), mean(y == "spam")))

idx <- createDataPartition(y, p = 0.75, list = FALSE)
Xtr <- X[idx, ]; Xte <- X[-idx, ]; ytr <- y[idx]; yte <- y[-idx]
dtr <- cbind(Xtr, type = ytr)
cat(sprintf("Apprentissage : %d | Test : %d\n", nrow(Xtr), nrow(Xte)))

Dimensions : 4601 x 57 | proportion de spam : 0.394


Apprentissage : 3451 | Test : 1150


### Observation des données

Avant d'entraîner le moindre modèle, on regarde les données : **variable cible** (équilibre des classes),
**distribution** des variables explicatives et **corrélations**. Ce sont ces constats qui justifient les
choix de prétraitement et de métriques qui suivent.

#### 1. La variable cible

In [7]:
effectifs <- table(y)
print(data.frame(Effectif = as.vector(effectifs), Proportion = round(as.vector(prop.table(effectifs)), 3),
                 row.names = names(effectifs)))
cat("Accuracy du classifieur 'toujours nonspam' :", round(mean(y == "nonspam"), 3), "\n")

plot_ly(x = names(effectifs), y = as.vector(effectifs), type = "bar",
        marker = list(color = c(pick_color(0), pick_color(1)))) %>%
  layout(title = "Repartition de la variable cible", yaxis = list(title = "Nombre d'emails")) %>%
  apply_dark_theme()

        Effectif Proportion
nonspam     2788      0.606
spam        1813      0.394


Accuracy du classifieur 'toujours nonspam' : 0.606 


HTML widgets cannot be represented in plain text (need html)

**Lecture.** Environ **39 % de spams** pour 61 % d'emails légitimes : les classes sont **déséquilibrées sans
être extrêmes**. Deux conséquences :
- un classifieur « idiot » qui répond toujours *non-spam* a déjà une **accuracy de 0.606** : c'est la **référence
  minimale** à battre, et la raison pour laquelle l'accuracy seule est trompeuse ;
- on **stratifie** le découpage apprentissage / test (et les plis de validation croisée) pour conserver cette
  proportion partout.

#### 2. Les variables explicatives

Les 57 variables sont de trois types : **48 fréquences de mots** (`make`, `free`, `hp`…, en % des mots de
l'email), **6 fréquences de caractères** (`;`, `(`, `[`, `!`, `$`, `#`) et **3 longueurs de séquences de
majuscules** (moyenne, plus longue, total).

In [8]:
cat("Valeurs manquantes :", sum(is.na(X)), "\n")
asymetrie <- function(v) mean((v - mean(v))^3) / sd(v)^3
resume <- data.frame(
  pct_zeros = colMeans(X == 0) * 100,
  mediane   = apply(X, 2, median),
  max       = apply(X, 2, max),
  asymetrie = apply(X, 2, asymetrie)
)
cat("Mediane du % de zeros :", median(resume$pct_zeros), "\n")
head(round(resume[order(-resume$asymetrie), ], 2), 10)

Valeurs manquantes : 0 


Mediane du % de zeros : 83.6992 


,pct_zeros,mediane,max,asymetrie
,<dbl>,<dbl>,<dbl>,<dbl>
charHash,83.70,0.00,19.83,31.04
capitalLong,0.00,15.00,9989.00,30.74
parts,98.20,0.00,8.33,28.24
num3d,98.98,0.00,42.81,26.21
capitalAve,0.00,2.28,1102.50,23.75
charSquarebracket,88.50,0.00,4.08,21.07
table,98.63,0.00,2.17,19.85
conference,95.59,0.00,10.00,19.71
project,92.89,0.00,20.00,18.76


**Lecture.**
- **Aucune valeur manquante**, mais énormément de **zéros** : la plupart des mots sont absents de la plupart des
  emails (la médiane du % de zéros est d'environ 84 %), d'où des **médianes nulles**.
- Les distributions sont **très asymétriques** (coefficient d'asymétrie de 20 à 30) avec des **valeurs extrêmes**
  (une séquence de majuscules de près de 10 000 caractères).
- Les **échelles sont très différentes** : fréquences entre 0 et 100, longueurs de majuscules jusqu'à plusieurs
  milliers.

Conséquences : l'hypothèse **gaussienne** de LDA/QDA est clairement fausse ; le **k-NN** doit être précédé d'une
**standardisation** (sinon les longueurs de majuscules écrasent toutes les autres variables) ; les **arbres**,
eux, sont insensibles à l'échelle et aux valeurs extrêmes.

#### 3. Corrélations

**Avec la cible.** On calcule la corrélation (de Pearson) entre chaque variable et l'indicatrice « spam ».

In [9]:
y01 <- as.numeric(y == "spam")
corr_cible <- sapply(X, cor, y = y01)
top <- head(corr_cible[order(-abs(corr_cible))], 10)
plot_ly(x = top, y = factor(names(top), levels = rev(names(top))), type = "bar", orientation = "h",
        marker = list(color = pick_color(0))) %>%
  layout(title = "Top 10 des correlations avec la cible", xaxis = list(title = "Correlation avec spam")) %>%
  apply_dark_theme()

HTML widgets cannot be represented in plain text (need html)

In [10]:
# Distribution par classe des 4 variables les plus liees a la cible (log(1+x) a cause de l'asymetrie)
vars <- names(top)[1:4]
long <- data.frame(variable = rep(vars, each = nrow(X)),
                   valeur   = log1p(unlist(X[vars])),
                   classe   = rep(y, times = length(vars)))
plot_ly(long, x = ~variable, y = ~valeur, color = ~classe, colors = c(pick_color(0), pick_color(1)),
        type = "box") %>%
  layout(boxmode = "group", title = "Variables les plus liees a la cible, par classe",
         yaxis = list(title = "log(1 + x)")) %>%
  apply_dark_theme()

HTML widgets cannot be represented in plain text (need html)

**Lecture.** Les mots les plus liés au spam sont ceux du marketing (`your`, `000`, `remove`, `free`,
`business`) et le caractère `$`. À l'inverse, `hp` (Hewlett-Packard, où le corpus a été collecté) est un
marqueur d'email **légitime** (corrélation négative). Mais **aucune corrélation ne dépasse 0.4** : aucune variable
ne suffit seule, il faut **combiner** beaucoup de variables, ce que font les modèles multivariés.

⚠️ La corrélation de Pearson ne mesure qu'une liaison **linéaire** et est sensible aux valeurs extrêmes : sur des
variables aussi asymétriques, les boîtes à moustaches en échelle $\log(1+x)$ sont plus parlantes.

**Entre variables explicatives.** Des variables très corrélées entre elles sont **redondantes**.

In [11]:
corr <- cor(X)
plot_ly(x = colnames(corr), y = colnames(corr), z = corr, type = "heatmap",
        colorscale = DIVERGING_SCALE, zmin = -1, zmax = 1) %>%
  layout(title = "Correlations entre les 57 variables") %>%
  apply_dark_theme(height = 750)

HTML widgets cannot be represented in plain text (need html)

In [12]:
# Paires les plus correlees (triangle superieur pour ne pas compter chaque paire deux fois)
haut <- which(upper.tri(corr), arr.ind = TRUE)
paires <- data.frame(var1 = rownames(corr)[haut[, 1]], var2 = colnames(corr)[haut[, 2]],
                     correlation = round(corr[haut], 3))
head(paires[order(-abs(paires$correlation)), ], 8)

,var1,var2,correlation
,<chr>,<chr>,<dbl>
560,num857,num415,0.996
773,num857,direct,0.848
775,num415,direct,0.845
496,telnet,num857,0.738
559,telnet,num415,0.735
627,num857,technology,0.730
629,num415,technology,0.727
772,telnet,direct,0.700


**Lecture.** La plupart des corrélations sont faibles, mais un **bloc** de variables est fortement lié : `857`,
`415`, `direct`, `telnet`, `technology` (en particulier $r \approx 0.996$ entre `857` et `415`, quasiment la
même variable). Ce sont des numéros de téléphone / termes techniques présents dans les mêmes emails internes.

Impact sur les modèles :
- **LDA** : covariance quasi singulière → coefficients **instables et non interprétables** (la prédiction reste
  correcte) ;
- **QDA** : une covariance **par classe** à inverser, encore plus fragile → **régularisation** nécessaire ;
- **k-NN** : une information redondante **compte plusieurs fois** dans la distance ;
- **arbres / forêts** : peu gênés pour prédire, mais l'**importance** se répartit entre variables corrélées.

### Rappel — Points d'attention pour chaque modèle

| Modèle | Principe | Points d'attention (en particulier sur `spambase`) |
|---|---|---|
| **LDA** | Gaussiennes de **même** covariance → frontière **linéaire** | Hypothèse gaussienne fausse ici (zéros, asymétrie) ; sensible aux **valeurs extrêmes** et à la **colinéarité** (`857`/`415`) ; pas d'hyperparamètre ; invariante à l'échelle (pas besoin de standardiser). |
| **QDA** | Une covariance **par classe** → frontière **quadratique** | ≈ 1 650 paramètres de covariance **par classe** : matrices quasi singulières (variables presque toujours nulles dans une classe) → **régulariser** (en R, `MASS::qda` ne régularise pas : surveiller les avertissements de rang) ; probabilités très extrêmes (0 ou 1) → mal calibrées, accuracy parfois décevante malgré une AUC correcte. |
| **CART** | Découpages successifs par seuils sur une variable | **Surapprentissage** si l'arbre est trop profond → régler profondeur / taille minimale des feuilles / `cp` par **validation croisée** ; très **instable** ; insensible à l'échelle et aux valeurs extrêmes (aucun prétraitement) ; probabilités = proportions dans les feuilles → peu de valeurs distinctes, ROC « en escalier ». |
| **CART coût 5:1** | Même arbre, erreurs sur les spams pénalisées 5× | Modifie l'**arbre lui-même** et le point de fonctionnement, pas la capacité à ordonner → l'évaluer sur le **rappel / le coût**, pas sur l'accuracy ; il « sacrifie » volontairement de l'accuracy. |
| **Forêt aléatoire** | Moyenne de nombreux arbres décorrélés (bootstrap + tirage de variables) | Hyperparamètre principal : nombre de variables tirées (`max_features` / `mtry`) ; **plus d'arbres** ne fait pas surapprendre (juste plus lent) ; probabilités « tassées » vers 0.5 → seuil théorique à manier avec prudence ; importances partagées entre variables corrélées ; peu interprétable. |
| **k-NN** | Vote des $k$ plus proches voisins | **Standardisation obligatoire** (échelles de 0–100 à plusieurs milliers), apprise sur le **train uniquement** (`Pipeline` / `preProcess`) ; $k$ par validation croisée ; 57 dimensions → **fléau de la dimension** ; variables redondantes comptées plusieurs fois ; probabilités multiples de $1/k$ ; prédiction coûteuse. |

**Pour tous les modèles :** les hyperparamètres se règlent par **validation croisée sur l'apprentissage** ; le jeu
de **test** ne sert qu'**une fois**, à la fin, pour comparer.

On construit les modeles demandes. Les hyperparametres sensibles (profondeur d'arbre, `k`) sont regles par
**validation croisee** (`caret`) ; LDA/QDA n'en ont quasiment pas. Le **k-NN** est precede d'une
**standardisation**. Le **CART a cout 5:1** utilise une **matrice de perte** penalisant 5x le fait de **rater un
spam**.


In [13]:
ctrl <- trainControl(method = "cv", number = 5, classProbs = TRUE,
                     summaryFunction = twoClassSummary)

# LDA / QDA : peu d'hyperparametres
mod_lda <- lda(Xtr, ytr)
mod_qda <- qda(Xtr, ytr)

# CART : profondeur/complexite reglees par validation croisee interne de rpart
mod_cart <- train(type ~ ., data = dtr, method = "rpart",
                  tuneLength = 10, trControl = ctrl, metric = "ROC")

# CART a cout differencie 5:1 : rater un spam coute 5x plus qu'une fausse alerte.
# loss[i, j] = cout de predire j quand la vraie classe est i (ordre : nonspam, spam)
L <- matrix(c(0, 1,
              5, 0), nrow = 2, byrow = TRUE)
mod_cart_cout <- rpart(type ~ ., data = dtr, method = "class", parms = list(loss = L))

# Foret aleatoire
mod_rf <- ranger(type ~ ., data = dtr, num.trees = 300, probability = TRUE)

# k-NN precede d'une standardisation, k regle par validation croisee
mod_knn <- train(Xtr, ytr, method = "knn", preProcess = c("center", "scale"),
                 tuneGrid = data.frame(k = seq(3, 21, 2)),
                 trControl = ctrl, metric = "ROC")
cat("k-NN : k optimal =", mod_knn$bestTune$k, "\n")
cat("Modeles entraines : LDA, QDA, CART, CART cout 5:1, Foret aleatoire, k-NN\n")

k-NN : k optimal = 15 


Modeles entraines : LDA, QDA, CART, CART cout 5:1, Foret aleatoire, k-NN


### Comparaison finale sur le jeu de test : ROC, AUC et autres métriques

In [14]:
auc_of <- function(prob)
  as.numeric(roc(yte, prob, levels = c("nonspam", "spam"), direction = "<", quiet = TRUE)$auc)
acc_of <- function(pred) mean(pred == yte)

probs <- list(
  "LDA"             = predict(mod_lda, Xte)$posterior[, "spam"],
  "QDA"             = predict(mod_qda, Xte)$posterior[, "spam"],
  "CART"            = predict(mod_cart, Xte, type = "prob")[, "spam"],
  "CART cout 5:1"   = predict(mod_cart_cout, Xte)[, "spam"],
  "Foret aleatoire" = predict(mod_rf, Xte)$predictions[, "spam"],
  "k-NN"            = predict(mod_knn, Xte, type = "prob")[, "spam"]
)
preds <- list(
  "LDA"             = predict(mod_lda, Xte)$class,
  "QDA"             = predict(mod_qda, Xte)$class,
  "CART"            = predict(mod_cart, Xte),
  "CART cout 5:1"   = predict(mod_cart_cout, Xte, type = "class"),
  "Foret aleatoire" = factor(ifelse(predict(mod_rf, Xte)$predictions[, "spam"] >= 0.5, "spam", "nonspam"),
                             levels = c("nonspam", "spam")),
  "k-NN"            = predict(mod_knn, Xte)
)

# Precision, rappel et F1 sur la classe positive "spam"
metriques <- function(pred) {
  vp <- sum(pred == "spam" & yte == "spam")
  fp <- sum(pred == "spam" & yte == "nonspam")
  fn <- sum(pred == "nonspam" & yte == "spam")
  c(Precision = vp / (vp + fp), Rappel = vp / (vp + fn), F1 = 2 * vp / (2 * vp + fp + fn))
}

fig <- plot_ly()
res <- data.frame()
i <- 0
for (nom in names(probs)) {
  pr <- probs[[nom]]
  r  <- roc(yte, pr, levels = c("nonspam", "spam"), direction = "<", quiet = TRUE)
  fig <- fig %>% add_trace(x = 1 - r$specificities, y = r$sensitivities,
                           type = "scatter", mode = "lines",
                           line = list(width = 3, color = pick_color(i)),
                           name = sprintf("%s (AUC=%.3f)", nom, as.numeric(r$auc)))
  m  <- metriques(preds[[nom]])
  res <- rbind(res, data.frame(Modele = nom, Accuracy = acc_of(preds[[nom]]), Precision = m[["Precision"]],
                               Rappel = m[["Rappel"]], F1 = m[["F1"]], AUC = auc_of(pr)))
  i <- i + 1
}
fig <- fig %>%
  add_trace(x = c(0, 1), y = c(0, 1), type = "scatter", mode = "lines",
            line = list(color = MUTED, dash = "dash"), name = "Aleatoire") %>%
  layout(title = "Detection de spam : comparaison des modeles (ROC, test)",
         xaxis = list(title = "Taux de faux positifs"),
         yaxis = list(title = "Taux de vrais positifs")) %>%
  apply_dark_theme()
fig

HTML widgets cannot be represented in plain text (need html)

In [15]:
res <- res[order(-res$AUC), ]
res[, -1] <- round(res[, -1], 3)
rownames(res) <- NULL
res

Modele,Accuracy,Precision,Rappel,F1,AUC
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Foret aleatoire,0.951,0.932,0.945,0.939,0.986
k-NN,0.897,0.881,0.852,0.866,0.956
LDA,0.888,0.897,0.808,0.850,0.951
QDA,0.840,0.720,0.971,0.827,0.948
CART,0.911,0.903,0.868,0.885,0.927
CART cout 5:1,0.853,0.738,0.971,0.839,0.911


### Comparaison des métriques : accuracy, F1 et AUC

| Métrique | Formule | Dépend du seuil ? | Ce qu'elle mesure | Limites |
|---|---|---|---|---|
| **Accuracy** | $\dfrac{VP + VN}{n}$ | oui | proportion de bonnes décisions | dépend de la **proportion de classes** (0.606 sans rien faire ici) ; traite FP et FN de la même façon |
| **F1** | $\dfrac{2\,\text{Précision}\cdot\text{Rappel}}{\text{Précision}+\text{Rappel}} = \dfrac{2VP}{2VP + FP + FN}$ | oui | qualité de détection de la **classe positive** (spam) | ignore les VN ; dépend de la classe choisie comme positive ; pondère précision et rappel à égalité (pas de coût 5:1) |
| **AUC** | $\mathbb{P}\big(s(X^+) > s(X^-)\big)$ | **non** | qualité de l'**ordonnancement** des scores | ne dit pas quel seuil choisir ; insensible à la calibration des probabilités ; ignore les coûts |

In [16]:
fig <- plot_ly()
for (k in 1:3) {
  metrique <- c("Accuracy", "F1", "AUC")[k]
  fig <- fig %>% add_trace(x = res$Modele, y = res[[metrique]], type = "bar", name = metrique,
                           marker = list(color = pick_color(k - 1)))
}
fig %>%
  add_segments(x = res$Modele[1], xend = res$Modele[nrow(res)], y = mean(yte == "nonspam"),
               yend = mean(yte == "nonspam"), name = "'toujours nonspam'",
               line = list(dash = "dash", color = MUTED)) %>%
  layout(barmode = "group", title = "Accuracy, F1 et AUC par modele (test)",
         xaxis = list(categoryorder = "array", categoryarray = res$Modele), yaxis = list(range = c(0.5, 1))) %>%
  apply_dark_theme()

HTML widgets cannot be represented in plain text (need html)

**Lecture.**
- Tous les modèles battent nettement la référence « toujours non-spam » (0.606), et la **forêt aléatoire** est
  la meilleure sur les trois métriques.
- Les **classements** par accuracy / F1 et par AUC ne coïncident **pas** : le **CART** a la 2ᵉ meilleure accuracy
  et le 2ᵉ meilleur F1, mais une AUC **inférieure à celle de la LDA**. Ses probabilités ne prennent que quelques
  valeurs (une par feuille) : il **décide** bien au seuil 1/2 mais **ordonne** mal les emails entre eux.
- La **QDA** a un rappel très élevé (≈ 0.97) mais une précision faible (≈ 0.73) : elle classe trop d'emails en
  spam car ses probabilités sont extrêmes et mal calibrées ; son AUC reste correcte car l'**ordre** des scores
  est bon.
- Le **CART coût 5:1** fait exactement ce qu'on lui demande : rappel ↑, précision ↓, accuracy ↓. Aucune de ces
  trois métriques n'intègre le coût 5:1 : c'est l'objet du complément plus bas.
- Le **F1** est plus sévère que l'accuracy car il ne profite pas des nombreux VN (emails légitimes faciles).

**Influence du seuil.** Pour la forêt aléatoire, on fait varier le seuil de décision : accuracy et F1 bougent,
l'AUC reste fixe (elle ne dépend que de l'ordre des scores).

In [17]:
p_rf <- probs[["Foret aleatoire"]]
seuils <- seq(0.05, 0.95, by = 0.05)
acc_s <- sapply(seuils, function(s) mean((p_rf >= s) == (yte == "spam")))
f1_s  <- sapply(seuils, function(s) metriques(ifelse(p_rf >= s, "spam", "nonspam"))[["F1"]])
plot_ly(x = seuils, y = acc_s, type = "scatter", mode = "lines", name = "Accuracy",
        line = list(width = 3, color = pick_color(0))) %>%
  add_trace(y = f1_s, name = "F1", line = list(width = 3, color = pick_color(1))) %>%
  add_trace(y = rep(auc_of(p_rf), length(seuils)), name = "AUC", line = list(dash = "dash", color = HIGHLIGHT)) %>%
  layout(title = "Foret aleatoire : metriques selon le seuil de decision",
         xaxis = list(title = "Seuil sur P(spam | x)"), yaxis = list(title = "Valeur de la metrique")) %>%
  apply_dark_theme()

HTML widgets cannot be represented in plain text (need html)

**Lecture.** L'accuracy et le F1 **changent avec le seuil** (et ne sont pas forcément maximaux au même
endroit) ; l'AUC, elle, est une ligne horizontale. D'où la démarche : on **compare les modèles** avec l'AUC, puis
on **choisit le seuil** du modèle retenu selon le critère métier (accuracy, F1, ou coût $FP + 5\,FN$), sur des
données de validation.

### Complément — Coût différencié : matrices de confusion et choix du seuil

L'AUC compare les modèles **indépendamment du seuil**. Pour une **décision**, il faut fixer un seuil et regarder
la matrice de confusion. Avec un coût de 5 pour un spam raté (FN) et 1 pour un email légitime bloqué (FP), le
seuil optimal théorique est $s^\star = 1/6$ (voir l'annexe). On compare le **coût total** $FP + 5\,FN$ sur le
test pour plusieurs situations.

In [18]:
C_FP <- 1; C_FN <- 5
situations <- list(
  "CART (seuil 1/2)"          = probs[["CART"]] >= 0.5,
  "CART cout 5:1 (perte 5:1)" = preds[["CART cout 5:1"]] == "spam",
  "Foret (seuil 1/2)"         = probs[["Foret aleatoire"]] >= 0.5,
  "Foret (seuil 1/6)"         = probs[["Foret aleatoire"]] >= C_FP / (C_FP + C_FN)
)
est_spam <- yte == "spam"
tableau <- do.call(rbind, lapply(names(situations), function(nom) {
  p <- situations[[nom]]
  vp <- sum(p & est_spam); fn <- sum(!p & est_spam); fp <- sum(p & !est_spam); vn <- sum(!p & !est_spam)
  data.frame(Situation = nom, VN = vn, FP = fp, FN = fn, VP = vp,
             Sensibilite = round(vp / (vp + fn), 3), Specificite = round(vn / (vn + fp), 3),
             Cout = C_FP * fp + C_FN * fn)
}))
print(tableau, row.names = FALSE)

# Cout total de la foret selon le seuil
seuils <- seq(0.01, 0.99, by = 0.01)
p_rf <- probs[["Foret aleatoire"]]
couts <- sapply(seuils, function(s) C_FP * sum(p_rf >= s & !est_spam) + C_FN * sum(p_rf < s & est_spam))
plot_ly(x = seuils, y = couts, type = "scatter", mode = "lines", name = "Cout total (test)",
        line = list(width = 3, color = pick_color(0))) %>%
  add_segments(x = 1/6, xend = 1/6, y = min(couts), yend = max(couts), name = "s* = 1/6",
               line = list(dash = "dash", color = HIGHLIGHT)) %>%
  add_segments(x = 0.5, xend = 0.5, y = min(couts), yend = max(couts), name = "seuil 1/2",
               line = list(dash = "dot", color = MUTED)) %>%
  layout(title = "Foret aleatoire : cout FP + 5 FN selon le seuil de decision",
         xaxis = list(title = "Seuil sur P(spam | x)"), yaxis = list(title = "Cout total sur le test")) %>%
  apply_dark_theme()

                 Situation  VN  FP FN  VP Sensibilite Specificite Cout
          CART (seuil 1/2) 655  42 60 393       0.868       0.940  342
 CART cout 5:1 (perte 5:1) 541 156 13 440       0.971       0.776  221
         Foret (seuil 1/2) 666  31 25 428       0.945       0.956  156
         Foret (seuil 1/6) 556 141  6 447       0.987       0.798  171


HTML widgets cannot be represented in plain text (need html)

**Lecture.** Au seuil $1/2$, un modèle minimise le **nombre** d'erreurs, pas leur **coût**. Déplacer la décision
vers les spams (CART pondéré 5:1, ou seuil abaissé) augmente la **sensibilité** (FN ↓) au prix de fausses
alertes supplémentaires (spécificité ↓). Le tableau et la courbe montrent deux choses :
- pour un modèle **peu performant** (CART), la pondération 5:1 réduit fortement le coût : on échange beaucoup de
  FN coûteux contre des FP bon marché ;
- pour la **forêt**, déjà très bonne au seuil $1/2$, le seuil théorique $s^\star = 1/6$ n'est pas forcément
  meilleur : il suppose des probabilités **bien calibrées** ($\hat{\mathbb{P}}(Y=1\mid x) \approx \eta(x)$), ce qui
  n'est pas le cas des forêts (leurs probabilités sont « tassées » vers le centre). La courbe de coût est plate
  sur une large plage : le choix du seuil compte moins que la qualité du score.

⚠️ En pratique, le seuil doit être choisi sur des données de **validation** (ou par validation croisée), jamais
sur le test : ici la courbe sur le test sert uniquement d'**illustration**.

### Conclusion

- La **foret aleatoire** obtient la **meilleure AUC** sur `spam` : elle capture des interactions complexes tout
  en restant robuste. Le **CART** simple est en retrait (instable), le **k-NN** est bon apres standardisation.
- **LDA/QDA** fournissent une base lineaire/quadratique solide mais restent limitees par leurs hypotheses
  gaussiennes, peu realistes sur des frequences de mots tres asymetriques.
- Le **CART a cout 5:1** ne vise pas la meilleure AUC : il **deplace le compromis** pour reduire un type
  d'erreur precis (rater un spam), ce qui se lit sur la **forme** de sa courbe ROC / sa matrice de confusion
  plutot que sur l'AUC globale.
- On compare **objectivement** les modeles par leur **AUC** ; le choix final du **seuil** depend ensuite du
  **cout** relatif des faux positifs (bloquer un email legitime) et des faux negatifs (laisser passer un spam).


## Annexe — Pour aller plus loin sur les matrices de confusion, ROC et coût différencié

### 1. La matrice de confusion n’est qu’un seuil donné
La confusion résume les erreurs pour un **seuil précis**. Si on change le seuil, les valeurs de VP, FP, VN et FN changent. La ROC permet de voir ce qui se passe pour tous les seuils à la fois.

### 2. Pourquoi la ROC est utile ?
La courbe ROC trace le compromis entre :
- **TPR** = sensibilité = proportion de positifs bien détectés ;
- **FPR** = proportion de négatifs mal classés.

Un bon modèle est un modèle dont la courbe est proche du coin supérieur gauche : il détecte beaucoup de vrais positifs sans trop de faux positifs.

### 3. Qu’est-ce que l’AUC ?
L’AUC mesure la probabilité que, pour un positif et un négatif choisis au hasard, le score du positif soit plus élevé que celui du négatif.

- **AUC = 0.5** : performance aléatoire ;
- **AUC = 1** : modèle parfait ;
- **AUC proche de 1** : bon pouvoir de classement.

Autrement dit, l’AUC dit si le modèle classe bien les individus dans l’ordre correct, indépendamment du seuil retenu.

### 4. Pourquoi le coût différencié change le choix du seuil ?
Dans un problème de spam, un faux positif (bloquer un email légitime) et un faux négatif (laisser passer un spam) n’ont pas le même coût. On peut donc choisir un seuil qui favorise :
- la détection des spam (si le coût des faux négatifs est élevé) ;
- la protection des emails légitimes (si le coût des faux positifs est élevé).

La courbe ROC décrit le compromis, mais le seuil final dépend du contexte économique ou métier.

**Conclusion.** La ROC permet de comparer les modèles indépendamment du seuil, tandis que la matrice de confusion et le seuil choisi rendent le compromis explicite. C’est exactement la bonne distinction à faire entre « performance globale » et « décision opérationnelle ».

---

## Annexe — Seuil optimal quand les erreurs n'ont pas le même coût

On note $c_{FN}$ le coût d'un faux négatif (spam non détecté) et $c_{FP}$ celui d'un faux positif (email
légitime bloqué). En un point $x$, avec $\eta(x) = \mathbb{P}(Y = 1 \mid X = x)$ :
- prédire **spam** coûte en moyenne $c_{FP}\,(1 - \eta(x))$ (on se trompe si c'est un email légitime) ;
- prédire **non-spam** coûte en moyenne $c_{FN}\,\eta(x)$ (on se trompe si c'est un spam).

La règle de Bayes **à coûts** prédit donc spam dès que $c_{FN}\,\eta(x) \ge c_{FP}\,(1 - \eta(x))$, soit
$$\eta(x) \;\ge\; s^\star = \frac{c_{FP}}{c_{FP} + c_{FN}}.$$
- Coûts égaux : $s^\star = 1/2$ (on retrouve la règle de Bayes du TP1) ;
- coût 5:1 (rater un spam coûte 5 fois plus) : $s^\star = 1/(1+5) = 1/6$.

**Deux façons équivalentes** d'intégrer ce coût : (i) garder un modèle standard et **abaisser le seuil** à
$1/6$ sur ses probabilités ; (ii) **pondérer** les classes à l'apprentissage (`class_weight`, matrice `loss`
de `rpart`), ce qui déplace directement la décision. Dans les deux cas, l'AUC (qui ne dépend pas du seuil)
change peu : c'est le **point de fonctionnement** sur la courbe ROC qui se déplace.